# 02 – What do we lose by using 20 features instead of ~120?

The app uses 15 hand-picked inputs (+5 engineered) so a person can fill in the form and the reasons stay readable. This notebook measures the cost: the same LightGBM setup trained on every column of `application_train.csv`, on **the same applicants** in train and validation.

Protected attributes (gender, age, family status) are excluded from both, so the comparison is fair. Validation set only; the test set stays untouched.

In [1]:
import lightgbm as lgb
import pandas as pd
from sklearn.metrics import roc_auc_score

from credit_risk.config import ID_COL, TARGET
from credit_risk.data import find_train_file, load_splits
from credit_risk.models import fit_lgbm
from credit_risk.train import predict_default_proba

splits = load_splits()
curated = fit_lgbm(splits.train, splits.val)
curated_auc = roc_auc_score(splits.val[TARGET], predict_default_proba(curated, splits.val))
print(f"Curated (20 features): validation ROC-AUC {curated_auc:.4f}")

Curated (20 features): validation ROC-AUC 0.7584


In [2]:
# Every raw column except the ID, the target and the protected attributes.
raw = pd.read_csv(find_train_file()).set_index("SK_ID_CURR")
protected = ["CODE_GENDER", "DAYS_BIRTH", "NAME_FAMILY_STATUS"]
X_all = raw.drop(columns=["TARGET", *protected])
X_all["DAYS_EMPLOYED"] = X_all["DAYS_EMPLOYED"].mask(X_all["DAYS_EMPLOYED"] == 365243)
for col in X_all.select_dtypes(exclude="number").columns:
    X_all[col] = X_all[col].astype("category")

train_ids, val_ids = splits.train[ID_COL], splits.val[ID_COL]  # same applicants as above
model = lgb.LGBMClassifier(n_estimators=3000, learning_rate=0.03, num_leaves=31, min_child_samples=200,
                           subsample=0.8, subsample_freq=1, colsample_bytree=0.8, reg_lambda=1.0,
                           random_state=42, verbose=-1)
model.fit(X_all.loc[train_ids], raw.loc[train_ids, "TARGET"],
          eval_X=X_all.loc[val_ids], eval_y=raw.loc[val_ids, "TARGET"],
          callbacks=[lgb.early_stopping(200, verbose=False)])
all_auc = roc_auc_score(raw.loc[val_ids, "TARGET"], model.predict_proba(X_all.loc[val_ids])[:, 1])
print(f"All {X_all.shape[1]} columns: validation ROC-AUC {all_auc:.4f}")
print(f"Cost of the curated feature set: {all_auc - curated_auc:.4f} AUC")

All 117 columns: validation ROC-AUC 0.7525
Cost of the curated feature set: -0.0059 AUC


## Result

| Feature set | Validation ROC-AUC |
|---|---|
| Curated: 15 inputs + 5 engineered ratios | **0.758** |
| All 117 raw columns (no engineering) | 0.753 |

The curated set does not cost accuracy here; it is slightly better. The likely reason is the engineered
ratios (credit-to-annuity, payment-to-income, ...). A tree splits on one column at a time, so it can only
approximate a ratio of two columns with many small splits. Handing it the ratio directly is more efficient.

Caveats, to keep this honest:
- The all-columns model got no feature engineering and the same settings, not its own tuning.
- Kaggle's best scores (~0.80) also used six other tables (bureau history, past applications, payments).
  More *information*, not more columns of the same table, is what moves the needle.
- One split, one seed: a difference of 0.005 is within the range that a different split could change.

**Takeaway:** for this table, a small, readable feature set loses nothing, and it makes the form usable
and the reason codes understandable.